# Prepare first positive notes

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import pandas as pd

from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_diagnosis_label
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Patient-level BINARY CLASSIFICATION 1. In training set, find patients manually labeled as positive for SMM (n=199) 2. Identify the first positive SMM encournter 3. In testing set, find patients manually labeled as positive for SMM (n=85) 4. Identify the first positive SMM encournter


In [ ]:
readfile_name = paths.data("Update/Data/Testing/SMMNLP_testing_data.json")
dataset = pd.read_json(readfile_name)

In [ ]:
dataset.head(5)

In [ ]:
readfile_name = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
dataset_true = pd.read_excel(readfile_name)

In [ ]:
dataset_true.head(5)

In [ ]:
final = dataset.merge(
    dataset_true[["PatientSSN", "Note_index", "SMM Dx (final)"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)

## Identify the first positive SMM encournter


In [ ]:
llm_columns = ["SMM Dx (final)"]
final = add_parsed_labels(final, llm_columns, parse_diagnosis_label)

In [ ]:
notes_pos = final[final["post_SMM Dx (final)"] == "Yes"]

In [ ]:
len(notes_pos)

In [ ]:
notes_pos["NoteDate2"] = pd.to_datetime(notes_pos["NoteDate"], format="%d%b%Y")

In [ ]:
earliest_records = notes_pos.loc[notes_pos.groupby("PatientSSN")["NoteDate2"].idxmin()]

In [ ]:
earliest_records.head(10)

In [ ]:
len(earliest_records)

## Save the earliest note of each true positve patients


In [ ]:
savefile_name = paths.data("Update/Data/Testing/Outcome2_testing_dataset.json")
earliest_records.to_json(savefile_name, date_format="iso")

In [ ]:
savefile_name3 = paths.data("Update/Data/Testing/Outcome2_testing_dataset.xlsx")
earliest_records.to_excel(savefile_name3, engine="xlsxwriter", sheet_name="Sheet1", index=False)

## Generate the Reference Standard using: FirstCohort500_Complete.xlsx SecondCohort237_Phase2_True.xlsx


In [ ]:
readfile_name = paths.data("Update/Data/Testing/Outcome2_testing_dataset.json")
dataset = pd.read_json(readfile_name)

In [ ]:
dataset.head(5)

In [ ]:
len(dataset)

In [ ]:
truedata_s1 = paths.data("Random500/Update/FirstCohort500_Complete.xlsx")
true_s1 = pd.read_excel(truedata_s1, sheet_name="Phase2&3")[
    ["PatientSSN", "NoteDate", "NoteText", "HxNew SMM Dx", "UncertainFlag", "SMM Dx Date"]
]
truedata_s2 = paths.data("Second237/True/SecondCohort237_Phase2_True.xlsx")
true_s2 = pd.read_excel(truedata_s2)

In [ ]:
true_s1.head(5)

In [ ]:
true_s2.head(5)

In [ ]:
true_all = pd.concat([true_s1, true_s2], ignore_index=True)

In [ ]:
outcome2true = dataset.merge(
    true_all[["PatientSSN", "HxNew SMM Dx", "UncertainFlag", "SMM Dx Date"]],
    on=["PatientSSN"],
    how="inner",
)

In [ ]:
len(outcome2true)

In [ ]:
outcome2true.head(5)

In [ ]:
savefile_name3 = paths.data("Update/True/SMMNLP_testing_true_phase2.xlsx")
outcome2true.to_excel(savefile_name3, engine="xlsxwriter", sheet_name="Sheet1", index=False)